# Stage 2: Data Preprocessing, Scikit-Learn Pipelines & Leak-Free SMOTE
## Food Adulteration Risk Prediction Using Data Mining Techniques
**Course:** Data Mining Techniques (DMT), B.Tech Information Technology  
**Institution:** K. Ramakrishnan College of Technology (Autonomous), Trichy  
**Team:** Jayanth S (2403811720521023), Jithin AK (2403811720521025), Keerthivasan RP (2403811720521026)

---
> ⚠️ **CRITICAL DISCLAIMER: SYNTHETIC DATA — GENERATED FOR ACADEMIC DEMONSTRATION**  
> *This study uses domain-modeled synthetic data for academic evaluation. Not for real regulatory decision-making.*
---


### 1. Objective of Data Preprocessing Pipeline
Raw inspection records often contain noise, missing fields, severe outlier spikes (e.g. faulty sensors), and categorical attributes that machine learning algorithms cannot consume directly.
In this stage, we establish a robust, leak-free preprocessing pipeline:
1. **Duplicate Detection & Removal**
2. **Missing Value Imputation**: Median strategy for skewed physical metrics; Most Frequent strategy for categorical attributes.
3. **Outlier Mitigation**: Interquartile Range (IQR) clipping bounds ($[Q_1 - 1.5 	imes IQR, Q_3 + 1.5 	imes IQR]$) calculated solely on the training fold.
4. **Encoding**:
   - Ordinal standardization for 1-5 sensory scores (`color_score`, `smell_score`, `taste_score`, `packaging_quality_score`).
   - One-Hot Encoding for nominal categorical features (`food_category`, `preservatives_used`).
5. **Standardization**: Zero-mean, unit-variance scaling (`StandardScaler`) to prevent gradient dominance in distance-based algorithms (SVM, KNN).
6. **Train/Test Splitting**: Stratified 80/20 partitioning preserving positive class incidence.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append(os.path.abspath('..'))
from src.preprocessing import run_preprocessing_pipeline, create_preprocessor, IQRClipper, RANDOM_STATE

print(f"Reproducibility seed: {RANDOM_STATE}")


In [ ]:
# Load raw data
raw_path = '../data/raw/food_adulteration_raw.csv'
df_raw = pd.read_csv(raw_path)
print(f"Loaded raw records: {df_raw.shape}")
df_raw.info()


### 2. Theoretical & Practical Danger: Why SMOTE Before Split Leaks Data
A critical error in academic data mining projects is applying class rebalancing (e.g. SMOTE) on the entire dataset **prior to** train-test splitting.

#### Mathematical Explanation of Data Leakage:
1. **The Role of the Test Set:** The hold-out test set must emulate unseen, future operational food samples arriving at a checkpoint. It must reflect true real-world class distributions and empirical variances.
2. **How SMOTE Synthesizes Data:** SMOTE identifies the $k$-nearest minority neighbors of an instance $x_i$ and generates synthetic points along the connecting line segment:
   $$x_{new} = x_i + \lambda (x_{zi} - x_i), \quad \lambda \sim U(0, 1)$$
3. **The Leakage Mechanism:** If SMOTE is performed before partitioning:
   - A minority instance belonging to the future "test" set is used to synthesize a neighbor placed in the "training" set.
   - Or worse, a synthetic point in the test set is an interpolation between a training point and a test point.
4. **Consequences:** The classifier achieves an artificially inflated, optimistic test accuracy because it has already memorized interpolated representations of the test fold. In production, when genuine unseen samples arrive, accuracy plummets.

> 🛡️ **Methodological Rule:** All transformations, scalers, imputers, and SMOTE oversamplers must be fitted **EXCLUSIVELY** on the training set ($X_{train}$), and the test set ($X_{test}$) must remain completely un-resampled and untouched.


In [ ]:
# Execute the production pipeline
result = run_preprocessing_pipeline(
    raw_csv_path='../data/raw/food_adulteration_raw.csv',
    output_dir='../data/processed',
    model_dir='../models',
    random_state=RANDOM_STATE
)

df_train = result['df_train']
df_test = result['df_test']
preprocessor = result['preprocessor']

print("Preprocessing and split execution summary:")
print(f"Training shape after leak-free SMOTE: {df_train.shape} (50% Adulterated / 50% Safe)")
print(f"Testing shape (Pure, untouched distribution): {df_test.shape} ({df_test['is_adulterated'].mean():.2%} Adulterated)")


In [ ]:
# Verify balanced training class distribution vs pure test distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='is_adulterated', data=df_train, ax=axes[0], palette='Blues_r')
axes[0].set_title('Training Set Class Balance (After SMOTE)\nSYNTHETIC DATA — for academic demonstration')
axes[0].set_xticklabels(['Safe (0)', 'Adulterated (1)'])

sns.countplot(x='is_adulterated', data=df_test, ax=axes[1], palette='Oranges_r')
axes[1].set_title('Test Set Class Balance (Pure Hold-Out)\nSYNTHETIC DATA — for academic demonstration')
axes[1].set_xticklabels(['Safe (0)', 'Adulterated (1)'])

plt.tight_layout()
plt.show()


In [ ]:
# Display first 5 rows of fully preprocessed features
df_train.head()


### 3. Verification of Processed Artifacts
The processed files are now persisted in `data/processed/` and the serialized `preprocessor.joblib` is saved in `models/`.

These clean matrices will serve as direct inputs for **Feature Selection & Exploratory Data Analysis in Notebook 03**.
